# Original 데이터로 전체 재학습 및 추론

## 🎯 목표
- Original 데이터만 사용하여 전체 데이터셋으로 재학습
- 최종 모델을 test.csv에 대해 추론 수행
- 결과를 submission.csv로 저장

## 📊 실험 설계
- **훈련 데이터**: Original 데이터만 사용 (type == 'original')
- **검증 데이터**: Original 데이터의 20% 분할
- **테스트 데이터**: test.csv (라벨 없음)
- **모델**: KLUE-BERT-base
- **손실 함수**: Cross Entropy Loss


## Library Import


In [1]:
# 한국어 텍스트 감정 분석을 위한 필수 라이브러리들
from collections import Counter
import os
import platform
import re
import sys
import warnings

import matplotlib.pyplot as plt
plt.rc("font", family="NanumBarunGothic")

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import koreanize_matplotlib

# 머신러닝 관련 라이브러리
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# 트랜스포머 및 BERT 관련 라이브러리
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    set_seed,
    Trainer,
    TrainingArguments,
)

# PyTorch 데이터 처리
from torch.utils.data import Dataset

# 경고 메시지 필터링
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

# 라이브러리 버전 정보 출력
print("=== 라이브러리 버전 정보 ===")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {__import__('transformers').__version__}")
print(f"sklearn: {__import__('sklearn').__version__}")

# GPU 사용 가능 여부 확인
print("\n=== PyTorch GPU 지원 정보 ===")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU: {torch.cuda.current_device()}")
    print(f"GPU 이름: {torch.cuda.get_device_name()}")
else:
    print("CPU에서 실행 중")


=== 라이브러리 버전 정보 ===
Python: 3.10.13
Platform: Linux-5.4.0-99-generic-x86_64-with-glibc2.31
pandas: 2.2.3
numpy: 2.2.6
torch: 2.6.0+cu124
transformers: 4.55.0
sklearn: 1.6.1

=== PyTorch GPU 지원 정보 ===
CUDA 사용 가능: True
CUDA 버전: 12.4
GPU 개수: 1
현재 GPU: 0
GPU 이름: Tesla V100-SXM2-32GB


In [2]:
# 랜덤 시드 설정
RANDOM_STATE = 42
set_seed(RANDOM_STATE)

print(f"랜덤 시드 {RANDOM_STATE}로 설정 완료")


랜덤 시드 42로 설정 완료


In [3]:
# 데이터 로드
print("=== 데이터 로드 ===")
df_train = pd.read_csv("../../../data/raw/train.csv")
df_test = pd.read_csv("../../../data/raw/test.csv")

print(f"훈련 데이터 크기: {len(df_train):,}개")
print(f"테스트 데이터 크기: {len(df_test):,}개")
print(f"훈련 데이터 컬럼: {list(df_train.columns)}")
print(f"테스트 데이터 컬럼: {list(df_test.columns)}")

# Original 데이터만 필터링
df_original = df_train[df_train['type'] == 'original'].copy()
print(f"\nOriginal 데이터 크기: {len(df_original):,}개")
print(f"전체 훈련 데이터 대비 비율: {len(df_original)/len(df_train)*100:.1f}%")

# 라벨 분포 확인
print("\nOriginal 데이터 라벨 분포:")
label_counts = df_original['label'].value_counts().sort_index()
for label, count in label_counts.items():
    print(f"라벨 {label}: {count:,}개 ({count/len(df_original)*100:.1f}%)")

print("\n처음 5행:")
df_original.head()


=== 데이터 로드 ===
훈련 데이터 크기: 279,650개
테스트 데이터 크기: 59,928개
훈련 데이터 컬럼: ['ID', 'review', 'label', 'type']
테스트 데이터 컬럼: ['ID', 'review']

Original 데이터 크기: 139,825개
전체 훈련 데이터 대비 비율: 50.0%

Original 데이터 라벨 분포:
라벨 0: 57,033개 (40.8%)
라벨 1: 13,608개 (9.7%)
라벨 2: 49,708개 (35.6%)
라벨 3: 19,476개 (13.9%)

처음 5행:


,ID,review,label,type
1,1,어느 부잣집 도련님의 철없는 행각,1,original
2,3,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네,0,original
4,5,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...,0,original
5,6,주인공처럼 군생활하면 맞아죽는다진짜;,0,original
6,8,제발 사라가 satc만한 영화에 출연할 날은 언제일까?,0,original


In [4]:
# 데이터 전처리를 위한 복사본 생성
df_processed = df_original[["ID", "label", "review", "type"]].copy()
df_processed_train = df_test[["ID", "review"]].copy()
print(f"원본 데이터셋 크기: {len(df_processed):,}개")

df_processed.head()

원본 데이터셋 크기: 139,825개


,ID,label,review,type
1,1,1,어느 부잣집 도련님의 철없는 행각,original
2,3,0,왜이렇게 재미가없냐 원도 별로였지만 원보다 더 재미없네,original
4,5,0,참나ㅋㅋ이게 무슨 드라마 최초 뮤지컬드라마야 이게무슨ㅋㅋ걍 다른 드라마랑 똑같구만 ...,original
5,6,0,주인공처럼 군생활하면 맞아죽는다진짜;,original
6,8,0,제발 사라가 satc만한 영화에 출연할 날은 언제일까?,original


In [5]:
print("텍스트 정규화 과정 시작")
print("=" * 50)

# 대소문자 정규화 (BERT가 처리하지만 일관성을 위해)
print("\n1단계: 대소문자 정규화 수행 중...")
df_processed["review_normalized"] = df_processed["review"].str.lower()
df_processed_train["review_normalized"] = df_processed_train["review"].str.lower()
print("✓ 소문자 변환 완료")

# 구두점 정규화
print("\n2단계: 구두점 정규화 수행 중...")


def normalize_punctuation(text):
    # NaN 또는 float 타입 체크
    if pd.isna(text) or not isinstance(text, str):
        return text
    
    # 여러 개의 구두점을 하나로 정규화
    text = re.sub(r"[.]{2,}", ".", text)
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[?]{2,}", "?", text)
    text = re.sub(r"[,]{2,}", ",", text)

    # 구두점 주변 공백 정리
    text = re.sub(r"\s+([.,!?])", r"\1", text)
    text = re.sub(r"([.,!?])\s+", r"\1 ", text)

    return text


df_processed["review_normalized"] = df_processed["review_normalized"].apply(
    normalize_punctuation
)
df_processed_train["review_normalized"] = df_processed_train["review_normalized"].apply(
    normalize_punctuation
)
print("✓ 구두점 정규화 완료")

# 특수문자 추가 정리
print("\n3단계: 특수문자 정리 수행 중...")


def clean_special_chars(text):
    # NaN 또는 float 타입 체크
    if pd.isna(text) or not isinstance(text, str):
        return text
    
    # URL 패턴 제거 (있는 경우)
    # URL 패턴 제거 (http/https 및 fragment 포함)
    text = re.sub(
        r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+#]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+",
        "",
        text,
    )
    text = re.sub(
        r"www\.[a-zA-Z0-9\-_~:/?#\[\]@!$&'()*+,;=.]+",
        "",
        text,
    )

    # '노려p://'와 같은 케이스에서 p://만 사라지게 하려면 바로 뒤에 오는 p:// 패턴만 지우는 것이 적합합니다.
    # 기존 정규식은 p:// 뒤에 반드시 '.'이 온다고 가정하여 오작동하며, 문자열을 너무 많이 지웁니다.
    # 아래처럼 p://만 삭제하는 쪽이 의도와 부합합니다.
    text = re.sub(r"p://", "", text)

    # 지정된 도메인 리스트 확장에 따라 도메인 패턴 제거
    tlds = [
        'com', 'net', 'org', 'co', 'kr', 'io', 'me', 'info', 'biz', 'tv', 'ai', 'app', 'dev',
        'xyz', 'us', 'uk', 'jp', 'cn', 'ru', 'site', 'store', 'online', 'top', 'tech', 'shop', 'cloud'
    ]
    tld_pattern = "|".join(tlds)
    # 'something.tld' 또는 'something.something.tld' 등, 최소한 도메인 알파벳이 마지막인 경우
    text = re.sub(
        rf"\b[a-zA-Z0-9\-_]+(?:\.[a-zA-Z0-9\-_]+)*\.({tld_pattern})\b",
        "", text)
    


    # 이메일 패턴 제거 (있는 경우)
    text = re.sub(r"\S+@\S+", "", text)

    # sweeti-potato@와 같이 '@'로 끝나는 이메일 fragment 제거
    text = re.sub(r"\b[\w\.\-]+@\b", "", text)
    # 이메일 username@ 처럼 @로 끝나는 형태 - username까지 같이 삭제
    text = re.sub(r"\b[\w\.\-]+@(?=\s|$)", "", text)

    # hsj1549@ 처럼 '@'로 끝나는 멘션 등도 한 번 더 처리
    text = re.sub(r"@\b", "", text)
    text = re.sub(r"\b@\b", "", text)
    text = re.sub(r"\b@\s", " ", text)

    # 멘션 패턴 제거 (있는 경우)
    text = re.sub(r"@\w+", "", text)

    # 과도한 공백 정리
    text = re.sub(r"\s+", " ", text)


    # 『내용』, 【내용】, 《내용》, ｢내용｣ 등 특수 괄호(전각 포함)로 둘러싸인 텍스트 제거 (여러 번 등장 가능하므로 반복적으로 모두 제거)
    # 대표적인 특수 괄호 모음: 『 』 【 】 《 》 「 」 〈 〉 ｢ ｣ “ ” ‘ ’
    # re.DOTALL 옵션으로 줄바꿈 포함 영역 지움, 반복 적용해서 모든 패턴 제거
    special_bracket_pattern = r"[『【《「〈｢“‘](.*?)[』】》」〉｣”’]"
    prev_text = None
    while prev_text != text:
        prev_text = text
        text = re.sub(special_bracket_pattern, "", text, flags=re.DOTALL)

    # 날짜 제거 (YYYY-MM-DD, YYYY/MM/DD, YY.MM.DD, YYYY.MM.DD, YYYY년MM월DD일, MM/DD, MM-DD, MM.DD 등)
    date_patterns = [
        r'\b\d{4}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 2021-12-30, 2021/12/30, 2021.12.30
        r'\b\d{2}[-/.]\d{1,2}[-/.]\d{1,2}\b',      # 21-12-30, 21.12.30, 21/12/30
        r'\b\d{1,2}[-/.]\d{1,2}\b',                # 12-30, 12.30, 12/30 (월일)
        r'\b\d{4}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 2021년 12월 30일, 2021년12월30일
        r'\b\d{2}년\s?\d{1,2}월\s?\d{1,2}일\b',     # 21년 12월 30일
        r'\b\d{4}년\s?\d{1,2}월\b',                # 2021년 12월 (년월)
        r'\b\d{4}년\b',                            # 2021년
    ]
    for pat in date_patterns:
        text = re.sub(pat, "", text)

    # 전화번호 제거 패턴 추가
    phone_patterns = [
        r'\b01[016789][ -]?\d{3,4}[ -]?\d{4}\b',        # 010-1234-5678, 011 222 3333, 01612345678 등
        r'\b\d{2,4}[ -]?\d{3,4}[ -]?\d{4}\b',           # 02-123-4567, 053 123 4567, 0311234567 등 일반 번호
        r'\b\d{4}[ -]?\d{4}\b',                         # 1234-5678, 12345678 등
    ]
    for pat in phone_patterns:
        text = re.sub(pat, "", text)

    # 금액 패턴 제거
    price_patterns = [
        r'\b\d+원\b',                    # 1000원, 50000원
        r'\b\d+,\d+원\b',               # 1,000원, 50,000원
        r'\b\d+\.\d+원\b',              # 1000.5원
        r'\b\d+만원\b',                 # 1만원, 10만원
        r'\b\d+천원\b',                 # 1천원, 5천원
        r'\b\d+억원\b',                 # 1억원
        r'\$\d+',                       # $100, $50
        r'\b\d+달러\b',                 # 100달러
    ]
    for pat in price_patterns:
        text = re.sub(pat, "", text)

    # 시간 패턴 제거
    time_patterns = [
        r'\b\d{1,2}:\d{2}(?::\d{2})?\b',  # 14:30, 14:30:25
        r'\b\d{1,2}시\s?\d{1,2}분\b',     # 2시 30분, 2시30분
        r'\b\d{1,2}시간\b',               # 2시간, 10시간
        r'\b\d{1,2}분\b',                 # 30분, 5분
        r'\b\d{1,2}초\b',                 # 30초, 5초
        r'\b오전\s?\d{1,2}시\b',          # 오전 9시
        r'\b오후\s?\d{1,2}시\b',          # 오후 2시
    ]  
    for pat in time_patterns:
        text = re.sub(pat, "", text)

    # 영화 관련 패턴 제거
    movie_patterns = [
        r'\b\d+편\b',                    # 1편, 2편, 3편
        r'\b\d+부작\b',                  # 1부작, 2부작
        r'\b\d+기\b',                    # 1기, 2기
        r'\b\d+회차\b',                  # 1회차, 2회차
        r'\b\d+화\b',                    # 1화, 2화
        r'\b\d+분\s?\d+초\b',            # 120분 30초
        r'\b\d+분\b',                    # 120분 (영화 상영시간)
        r'\b\d+등급\b',                  # 15등급, 18등급
        r'\b\d+세\s?이상\b',             # 15세 이상
    ]    
    for pat in movie_patterns:
        text = re.sub(pat, "", text)

    # SNS/플랫폼 패턴 제거
    sns_patterns = [
        r'\b#\w+\b',                     # 해시태그 #영화 #추천
        r'\b@\w+\b',                     # 멘션 @username
        r'\bRT\b',                       # 리트윗 표시
        r'\b좋아요\s?\d+\b',             # 좋아요 100
        r'\b댓글\s?\d+\b',               # 댓글 50
        r'\b공유\s?\d+\b',               # 공유 20
        r'\b조회수\s?\d+\b',             # 조회수 1000
        r'\b구독자\s?\d+\b',             # 구독자 5000
    ]
    for pat in sns_patterns:
        text = re.sub(pat, "", text)

    # 기타 노이즈 패턴 제거
    noise_patterns = [
        r'\b\d+번\b',                    # 1번, 2번
        r'\b\d+개\b',                    # 1개, 2개
        r'\b\d+명\b',                    # 1명, 2명
        r'\b\d+장\b',                    # 1장, 2장
        r'\b\d+회\b',                    # 1회, 2회
        r'\b\d+차\b',                    # 1차, 2차
        r'\b\d+번째\b',                  # 1번째, 2번째
        r'\b\d+위\b',                    # 1위, 2위
        r'\b\d+등\b',                    # 1등, 2등
        r'\b\d+점\b',                    # 1점, 2점
        r'\b\d+점대\b',                  # 1점대, 2점대
        r'\b\d+점만점\b',                # 10점만점
        r'\b\d+점\s?만점\b',             # 10점 만점
    ]
    for pat in noise_patterns:
        text = re.sub(pat, "", text)

    # 특수 문자 및 기호 제거
    special_chars = [
        r'[★☆♥♡♠♣♦]',                  # 특수 기호
        r'[♪♫♬♩]',                      # 음악 기호
        r'[→←↑↓]',                      # 화살표
        r'[①②③④⑤⑥⑦⑧⑨⑩]',            # 원 숫자
        r'[⑴⑵⑶⑷⑸⑹⑺⑻⑼⑽]',            # 괄호 숫자
        r'[❶❷❸❹❺❻❼❽❾❿]',            # 검은 원 숫자
        r'[ⓐⓑⓒⓓⓔⓕⓖⓗⓘⓙ]',            # 원 문자
    ]
    for pat in special_chars:
        text = re.sub(pat, "", text)

    return text.strip()


df_processed["review_normalized"] = df_processed["review_normalized"].apply(
    clean_special_chars
)
df_processed_train["review_normalized"] = df_processed_train["review_normalized"].apply(
    clean_special_chars
)
print("✓ URL/이메일/멘션 제거 완료")

# 정규화 후 빈 텍스트 처리
print("\n4단계: 정규화 후 빈 텍스트 확인 중...")
empty_after_normalization = df_processed["review_normalized"].str.strip().eq("").sum()
if empty_after_normalization > 0:
    df_processed = df_processed[df_processed["review_normalized"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_after_normalization}개 제거")
else:
    print("✓ 빈 텍스트 없음")

empty_after_normalization = df_processed_train["review_normalized"].str.strip().eq("").sum()
if empty_after_normalization > 0:
    df_processed_train = df_processed_train[df_processed_train["review_normalized"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_after_normalization}개 제거")
else:
    print("✓ 빈 텍스트 없음")


텍스트 정규화 과정 시작

1단계: 대소문자 정규화 수행 중...
✓ 소문자 변환 완료

2단계: 구두점 정규화 수행 중...
✓ 구두점 정규화 완료

3단계: 특수문자 정리 수행 중...
✓ URL/이메일/멘션 제거 완료

4단계: 정규화 후 빈 텍스트 확인 중...
✓ 빈 텍스트 135개 제거
✓ 빈 텍스트 44개 제거


In [6]:
# 기본 텍스트 정리 함수
def clean_text(text):
    """
    한국어 텍스트를 위한 기본 텍스트 정리 함수

    전처리 단계:
    1. 불완전한 한글 제거 (자음/모음만 있는 경우)
    2. 반복되는 감정 표현 정규화 (ㅋㅋㅋ, ㅠㅠㅠ 등)
    3. 과도한 문자 반복 축소 (4번 이상 → 3번으로)
    4. 특수문자 제거 (한글, 숫자, 기본 구두점, 감정표현 제외)
    5. 공백 정규화
    """
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # 한국어 특화 전처리
    text = re.sub(
        r"[ㄱ-ㅎㅏ-ㅣ]+", "", text
    )  # 불완전한 한글 제거 (자음/모음만 있는 경우)
    text = re.sub(r"([ㅋㅎ])\1{2,}", r"\1\1", text)  # 웃음 표현 정규화: ㅋㅋㅋ+ → ㅋㅋ
    text = re.sub(
        r"([ㅠㅜㅡ])\1{2,}", r"\1\1", text
    )  # 슬픔 표현 정규화: ㅠㅠㅠ+ → ㅠㅠ
    text = re.sub(
        r"(.)\1{3,}", r"\1\1\1", text
    )  # 과도한 반복 축소: 4번 이상 반복 → 3번으로
    text = re.sub(r"[^\w\s가-힣.,!?ㅋㅎㅠㅜㅡ~\-]", " ", text)  # 필요한 문자만 유지
    text = re.sub(r"\s+", " ", text)  # 다중 공백을 단일 공백으로

    return text.strip()


print("텍스트 전처리 과정 시작")
print("=" * 50)
initial_size = len(df_processed)
initial_size_train = len(df_processed_train)
print(f"초기 데이터 크기: {initial_size:,}개")
print(f"초기 데이터 크기: {initial_size_train:,}개")

# 1단계: 텍스트 정리 적용
print("\n1단계: 기본 텍스트 정리 수행 중...")
df_processed["review_cleaned"] = df_processed["review_normalized"].apply(clean_text)
df_processed_train["review_cleaned"] = df_processed_train["review_normalized"].apply(clean_text)
print("✓ 한글 자음/모음 정리, 반복 표현 정규화, 특수문자 제거 완료")

# 2단계: 빈 텍스트 제거
print("\n2단계: 빈 텍스트 제거 중...")
empty_count = df_processed["review_cleaned"].str.strip().eq("").sum()
if empty_count > 0:
    df_processed = df_processed[df_processed["review_cleaned"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_count}개 제거")
else:
    print("✓ 빈 텍스트 없음")

empty_count_train = df_processed_train["review_cleaned"].str.strip().eq("").sum()
if empty_count_train > 0:
    df_processed_train = df_processed_train[df_processed_train["review_cleaned"].str.strip() != ""]
    print(f"✓ 빈 텍스트 {empty_count_train}개 제거")
else:
    print("✓ 빈 텍스트 없음")


# 3단계: 중복 제거
print("\n3단계: 중복 데이터 제거 중...")
duplicates_count = df_processed.duplicated(subset=["review_cleaned", "label"]).sum()
if duplicates_count > 0:
    df_processed = df_processed.drop_duplicates(subset=["review_cleaned", "label"])
    print(f"✓ 중복 데이터 {duplicates_count}개 제거")
else:
    print("✓ 중복 데이터 없음")

duplicates_count_train = df_processed_train.duplicated(subset=["review_cleaned"]).sum()
if duplicates_count_train > 0:
    df_processed_train = df_processed_train.drop_duplicates(subset=["review_cleaned"])
    print(f"✓ 중복 데이터 {duplicates_count_train}개 제거")
else:
    print("✓ 중복 데이터 없음")

텍스트 전처리 과정 시작
초기 데이터 크기: 139,690개
초기 데이터 크기: 59,884개

1단계: 기본 텍스트 정리 수행 중...
✓ 한글 자음/모음 정리, 반복 표현 정규화, 특수문자 제거 완료

2단계: 빈 텍스트 제거 중...
✓ 빈 텍스트 345개 제거
✓ 빈 텍스트 151개 제거

3단계: 중복 데이터 제거 중...
✓ 중복 데이터 3891개 제거
✓ 중복 데이터 1314개 제거


In [7]:
class ReviewDataset(Dataset):
    """
    리뷰 데이터를 위한 PyTorch Dataset 클래스
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        """
        ReviewDataset 초기화
        
        Args:
            texts: 리뷰 텍스트 리스트 또는 pandas Series
            labels: 감정 라벨 리스트 또는 pandas Series (None이면 추론용)
            tokenizer: BERT 토크나이저
            max_length: 최대 시퀀스 길이 (기본값: 128)
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        """데이터셋 크기 반환"""
        return len(self.texts)
    
    def __getitem__(self, idx):
        """특정 인덱스의 데이터 아이템 반환"""
        # 텍스트 토크나이징 및 패딩
        encoding = self.tokenizer(
            str(self.texts.iloc[idx]) if hasattr(self.texts, 'iloc') else str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        
        # 기본 아이템 구성
        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }
        
        # 라벨 추가 (훈련용일 때만)
        if self.labels is not None:
            label = self.labels.iloc[idx] if hasattr(self.labels, 'iloc') else self.labels[idx]
            item["labels"] = torch.tensor(label, dtype=torch.long)
        
        return item

print("ReviewDataset 클래스 정의 완료!")


ReviewDataset 클래스 정의 완료!


In [10]:
# 훈련/검증 데이터 분할
print("=== 데이터 분할 ===")
train_data, val_data = train_test_split(
    df_processed,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_processed['label']  # 라벨 비율 유지
)

print(f"훈련 데이터: {len(train_data):,}개")
print(f"검증 데이터: {len(val_data):,}개")

# 라벨 분포 확인
print("\\n훈련 데이터 라벨 분포:")
train_label_counts = train_data['label'].value_counts().sort_index()
for label, count in train_label_counts.items():
    print(f"라벨 {label}: {count:,}개 ({count/len(train_data)*100:.1f}%)")

print("\\n검증 데이터 라벨 분포:")
val_label_counts = val_data['label'].value_counts().sort_index()
for label, count in val_label_counts.items():
    print(f"라벨 {label}: {count:,}개 ({count/len(val_data)*100:.1f}%)")


=== 데이터 분할 ===
훈련 데이터: 108,363개
검증 데이터: 27,091개
\n훈련 데이터 라벨 분포:
라벨 0: 44,366개 (40.9%)
라벨 1: 10,597개 (9.8%)
라벨 2: 38,226개 (35.3%)
라벨 3: 15,174개 (14.0%)
\n검증 데이터 라벨 분포:
라벨 0: 11,092개 (40.9%)
라벨 1: 2,649개 (9.8%)
라벨 2: 9,556개 (35.3%)
라벨 3: 3,794개 (14.0%)


In [11]:
# 모델 및 토크나이저 설정
model_name = "kykim/bert-kor-base"
print(f"🤖 모델 로딩: {model_name}")

# KLUE-BERT 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 시퀀스 분류를 위한 BERT 모델 로드 (4개 클래스 분류)
NUM_CLASSES = 4
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_CLASSES,
)

print(f"토크나이저 로드 완료: {tokenizer.__class__.__name__}")
print(f"모델 로드 완료: {model.__class__.__name__}")
print(f"분류 클래스 수: {NUM_CLASSES}")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")


🤖 모델 로딩: kykim/bert-kor-base


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at kykim/bert-kor-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


토크나이저 로드 완료: BertTokenizerFast
모델 로드 완료: BertForSequenceClassification
분류 클래스 수: 4
모델 파라미터 수: 118,300,420


In [12]:
# 데이터셋 생성
print("=== 데이터셋 생성 ===")

# 훈련 및 검증 데이터셋
train_dataset = ReviewDataset(
    train_data["review_cleaned"],
    train_data["label"],
    tokenizer,
    max_length=128
)

val_dataset = ReviewDataset(
    val_data["review_cleaned"],
    val_data["label"],
    tokenizer,
    max_length=128
)

# 테스트 데이터셋 (라벨 없음)
test_dataset = ReviewDataset(
    df_processed_train["review_cleaned"],
    None,  # 라벨 없음
    tokenizer,
    max_length=128
)

print(f"훈련 데이터셋 크기: {len(train_dataset):,}")
print(f"검증 데이터셋 크기: {len(val_dataset):,}")
print(f"테스트 데이터셋 크기: {len(test_dataset):,}")

# 데이터셋 샘플 확인
sample = train_dataset[0]
print(f"\\n훈련 데이터셋 샘플:")
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"label: {sample['labels'].item()}")

test_sample = test_dataset[0]
print(f"\\n테스트 데이터셋 샘플:")
print(f"input_ids shape: {test_sample['input_ids'].shape}")
print(f"attention_mask shape: {test_sample['attention_mask'].shape}")
print("라벨: 없음 (추론용)")


=== 데이터셋 생성 ===
훈련 데이터셋 크기: 108,363
검증 데이터셋 크기: 27,091
테스트 데이터셋 크기: 58,419
\n훈련 데이터셋 샘플:
input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: 3
\n테스트 데이터셋 샘플:
input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
라벨: 없음 (추론용)


In [13]:
def compute_metrics(eval_pred):
    """
    평가 지표 계산 함수
    
    Args:
        eval_pred: (predictions, labels) 튜플
    
    Returns:
        dict: 계산된 평가 지표들
    """
    predictions, labels = eval_pred
    
    # 예측값을 클래스 인덱스로 변환
    predictions = np.argmax(predictions, axis=1)
    
    # 정확도 계산
    accuracy = accuracy_score(labels, predictions)
    
    # F1 점수 계산 (macro average)
    f1 = f1_score(labels, predictions, average='macro')
    
    return {
        'accuracy': accuracy,
        'f1': f1
    }

print("평가 지표 함수 정의 완료!")


평가 지표 함수 정의 완료!


In [15]:
# 훈련 설정
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
SAVE_MODEL = True

print(f"=== 훈련 설정 ===")
print(f"배치 크기: {BATCH_SIZE}")
print(f"학습률: {LEARNING_RATE}")
print(f"에포크 수: {NUM_EPOCHS}")
print(f"모델 저장: {SAVE_MODEL}")

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir="./retraining_results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=LEARNING_RATE,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch" if SAVE_MODEL else "no",
    load_best_model_at_end=SAVE_MODEL,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2 if SAVE_MODEL else 0,
    seed=RANDOM_STATE,
    fp16=True,  # 혼합 정밀도 훈련
    dataloader_num_workers=4,
    remove_unused_columns=False,
)

print("TrainingArguments 설정 완료!")


=== 훈련 설정 ===
배치 크기: 32
학습률: 2e-05
에포크 수: 5
모델 저장: True
TrainingArguments 설정 완료!


In [16]:
# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Trainer 초기화 완료!")
print(f"훈련 샘플: {len(train_dataset):,}개")
print(f"검증 샘플: {len(val_dataset):,}개")

# 훈련 시작
print("=" * 50)
print("Original 데이터로 모델 훈련 시작")
print("=" * 50)

trainer.train()

print("\\n훈련 완료!")


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer 초기화 완료!
훈련 샘플: 108,363개
검증 샘플: 27,091개
Original 데이터로 모델 훈련 시작


wandb: Currently logged in as: pileuszu (pileuszu16) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.507900,0.502516,0.808387,0.736351
2,0.421300,0.481019,0.816987,0.749583
3,0.311900,0.556057,0.818095,0.750548
4,0.218600,0.612675,0.815843,0.753586
5,0.156900,0.719535,0.815954,0.756328


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

\n훈련 완료!


모델 최종 평가


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

\n최종 성능 결과:
정확도 (Accuracy): 0.8181
F1-Score: 0.7505
검증 손실 (Loss): 0.5561


테스트 데이터 추론 시작


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

테스트 데이터 예측 완료: 58,419개
예측된 라벨 분포:
라벨 0: 24,265개 (41.5%)
라벨 1: 4,526개 (7.7%)
라벨 2: 21,491개 (36.8%)
라벨 3: 8,137개 (13.9%)


ValueError: Length of values (58419) does not match length of index (59928)